In [1]:
import random
import shutil
from pathlib import Path
from sklearn.model_selection import train_test_split

MAX_IMAGES_PER_CLASS = 150
SEED = 42

SOURCE_DIR = Path("dataset/authentic")

OUTPUT_DIR = Path("dataset_split")

random.seed(SEED)

In [2]:
for split in ["train", "test"]:

    for cls_dir in SOURCE_DIR.iterdir():

        if cls_dir.is_dir():

            (OUTPUT_DIR / split / "authentic" /
             cls_dir.name).mkdir(
                parents=True,
                exist_ok=True
            )

In [3]:
for cls_dir in SOURCE_DIR.iterdir():

    if not cls_dir.is_dir():
        continue

    class_name = cls_dir.name

    images = list(cls_dir.glob("*"))

    random.shuffle(images)

    if len(images) > MAX_IMAGES_PER_CLASS:

        images = images[:MAX_IMAGES_PER_CLASS]

    train_imgs, test_imgs = train_test_split(
        images,
        test_size=0.20,
        random_state=SEED
    )

    splits = {
        "train": train_imgs,
        "test": test_imgs
    }

    for split_name, split_files in splits.items():

        dst = (
            OUTPUT_DIR /
            split_name /
            "authentic" /
            class_name
        )

        for img in split_files:

            shutil.copy2(
                img,
                dst / img.name
            )

print("Dataset split completed")

Dataset split completed


In [4]:
import cv2
import numpy as np

In [5]:
def copy_move_forgery(img):

    h, w = img.shape

    size = random.randint(32, 96)

    x1 = random.randint(0, w-size)
    y1 = random.randint(0, h-size)

    patch = img[
        y1:y1+size,
        x1:x1+size
    ].copy()

    x2 = random.randint(0, w-size)
    y2 = random.randint(0, h-size)

    forged = img.copy()

    forged[
        y2:y2+size,
        x2:x2+size
    ] = patch

    mask = np.zeros_like(img)

    mask[
        y2:y2+size,
        x2:x2+size
    ] = 255

    return forged, mask

In [6]:
def contrast_forgery(img):

    alpha = random.uniform(1.3, 2.0)

    beta = random.randint(-20, 20)

    forged = cv2.convertScaleAbs(
        img,
        alpha=alpha,
        beta=beta
    )

    mask = np.ones_like(img) * 255

    return forged, mask

In [7]:
def synthetic_defect(img):

    forged = img.copy()

    mask = np.zeros_like(img)

    n = random.randint(5, 15)

    for _ in range(n):

        r = random.randint(5, 15)

        x = random.randint(
            r,
            img.shape[1]-r
        )

        y = random.randint(
            r,
            img.shape[0]-r
        )

        cv2.circle(
            forged,
            (x,y),
            r,
            random.randint(180,255),
            -1
        )

        cv2.circle(
            mask,
            (x,y),
            r,
            255,
            -1
        )

    return forged, mask

In [8]:
def splice_forgery(img1, img2):

    h,w = img1.shape

    size = random.randint(64,128)

    xs = random.randint(0,w-size)
    ys = random.randint(0,h-size)

    patch = img2[
        ys:ys+size,
        xs:xs+size
    ]

    xd = random.randint(0,w-size)
    yd = random.randint(0,h-size)

    forged = img1.copy()

    forged[
        yd:yd+size,
        xd:xd+size
    ] = patch

    mask = np.zeros_like(img1)

    mask[
        yd:yd+size,
        xd:xd+size
    ] = 255

    return forged, mask

In [9]:
FORGERIES = [
    "copy_move",
    "splicing",
    "synthetic",
    "contrast"
]

In [10]:
for split in ["train","test"]:

    split_root = (
        OUTPUT_DIR /
        split /
        "authentic"
    )

    for class_dir in split_root.iterdir():

        if not class_dir.is_dir():
            continue

        cls = class_dir.name

        for forgery in FORGERIES:

            (
                OUTPUT_DIR /
                split /
                "forged" /
                forgery /
                cls
            ).mkdir(
                parents=True,
                exist_ok=True
            )

            (
                OUTPUT_DIR /
                split /
                "masks" /
                forgery /
                cls
            ).mkdir(
                parents=True,
                exist_ok=True
            )

In [11]:
for split in ["train","test"]:

    split_root = (
        OUTPUT_DIR /
        split /
        "authentic"
    )

    class_dirs = [
        d for d in split_root.iterdir()
        if d.is_dir()
    ]

    for cls_dir in class_dirs:

        image_list = list(
            cls_dir.glob("*")
        )

        for img_path in image_list:

            img = cv2.imread(
                str(img_path),
                cv2.IMREAD_GRAYSCALE
            )

            chosen = random.sample(
                FORGERIES,
                2
            )

            for forgery in chosen:

                if forgery == "copy_move":

                    forged, mask = \
                        copy_move_forgery(img)

                elif forgery == "contrast":

                    forged, mask = \
                        contrast_forgery(img)

                elif forgery == "synthetic":

                    forged, mask = \
                        synthetic_defect(img)

                elif forgery == "splicing":

                    donor_class = random.choice(
                        class_dirs
                    )

                    donor_img_path = random.choice(
                        list(
                            donor_class.glob("*")
                        )
                    )

                    donor = cv2.imread(
                        str(donor_img_path),
                        cv2.IMREAD_GRAYSCALE
                    )

                    donor = cv2.resize(
                        donor,
                        (
                            img.shape[1],
                            img.shape[0]
                        )
                    )

                    forged, mask = \
                        splice_forgery(
                            img,
                            donor
                        )
                save_name = (
                    f"{img_path.stem}"
                    f"_{forgery}.png"
                )
                
                forged_path = (
                    OUTPUT_DIR /
                    split /
                    "forged" /
                    forgery /
                    cls_dir.name /
                    save_name
                )
                
                mask_path = (
                    OUTPUT_DIR /
                    split /
                    "masks" /
                    forgery /
                    cls_dir.name /
                    save_name
                )
                
                cv2.imwrite(
                    str(forged_path),
                    forged
                )
                
                cv2.imwrite(
                    str(mask_path),
                    mask
                )                
print("Forgery generation complete")

Forgery generation complete
